Not all models are good at instruction following, and might require some post-processing to get to the appropriate structure. 

We make certain assumptions about the output format, and make simplifications to ensure that we can parse the output correctly.

For instance, if the expectation is to generate 4 hints, and the model generates 3, we repeat the last hint to ensure that we have 4 hints.

Different assumptions were made for different generation models, based on their output quality.

In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import os
import json
import tqdm
from nltk.tokenize import sent_tokenize

from generate_baseline_hints import clean_hint

output_dir = "output/baseline_generations"

## Static Hints

In [ ]:
# remove the gibberish preceeding the hints for the gemma-3 model outputs

for model_name in ["gemma3-1b", "gemma3-4b", "gemma3-12b", "gemma3-27b"]:
    print(f"Processing model: {model_name}")
    data_split = 'valid'
    experiment_name = f'{model_name}_static_{data_split}'
    hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

    # store the raw data into the raw directory
    raw_directory = "output/baseline_generations_raw"
    if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
        json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

    for ques, inst in hint_data.items():
        if inst[0][-1] != ':' and len(inst) > 4:
            inst[0] = inst[0] + ':'
        hints_str = ' '.join(inst).split(': *')[-1].strip()
        final_hints = hints_str.split(' * ')
        final_hints = [clean_hint(hint, 4) for hint in final_hints]

        assert len(final_hints) == 4, f"Expected 4 hints, found {len(inst)} for instance {ques}\n{final_hints}, {inst}"
        hint_data[ques] = final_hints

    # save the cleaned hints
    json.dump(hint_data, open(f'{output_dir}/{experiment_name}.json', 'w'), indent=4)

In [ ]:
# remove the gibberish preceeding the </think> tag from deepseek model family outputs

for model_name in ["deepseek-r1-1.5b", "deepseek-r1-7b", "deepseek-r1-8b", "deepseek-r1-14b", "deepseek-r1-32b"]:
    print(f"Processing model: {model_name}")
    data_split = 'valid'
    experiment_name = f"{model_name}_static_{data_split}"
    hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

    # store the raw data into the raw directory
    raw_directory = "output/baseline_generations_raw"
    if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
        json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

    # remove the thinking part of the generation
    for ques, inst in tqdm.tqdm(hint_data.items(), total=len(hint_data)):
        # final_hints = sent_tokenize('\n'.join(inst).split('</think>')[-1])
        final_hints = '\n'.join(inst).split('</think>')[-1].split('\n')
        if final_hints[-1] == '```' :
            final_hints = final_hints[:-1]
        if final_hints[0].startswith('Here are four') or final_hints[0].startswith('The following are four') or final_hints[0].startswith('Certainly') or final_hints[0].startswith('Here are 4 hints') or final_hints[0].startswith('Here are the') or final_hints[0].startswith('Four hints that'):
            final_hints = final_hints[1:]
        if final_hints[0] in ["Hints:", '**Hints:**', '**Hints**:', '```plaintext', 'hints', 'hints:']:
            final_hints = final_hints[1:]
        if (final_hints[-1].startswith('Each hint') or final_hints[-1].startswith('These hints') or final_hints[-1].startswith("5.")):
            final_hints = final_hints[:-1]
        if len(final_hints) == 3:
            final_hints.append(final_hints[2])
        final_hints = [clean_hint(hint, 4) for hint in final_hints]
        if len(final_hints) > 4:
            final_hints = final_hints[:4]
        if len(final_hints) < 4:
            while len(final_hints) < 4:
                final_hints.append(final_hints[-1])
        assert len(final_hints) == 4, f"Expected 4 hints, found {len(final_hints)} for instance {ques}\n{final_hints}"
        hint_data[ques] = final_hints
    
    # save the cleaned hints
    json.dump(hint_data, open(f'{output_dir}/{experiment_name}.json', 'w'), indent=4)

In [ ]:
# remove the gibberish preceeding the </think> tag from qwen model family outputs

for model_name in ["qwen3-0.6b", "qwen3-1.7b", "qwen3-4b", "qwen3-8b", "qwen3-14b", "qwen3-30b", "qwen3-32b"]:
    print(f"Processing model: {model_name}")
    data_split = 'valid'
    experiment_name = f"{model_name}_static_{data_split}"
    hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

    # store the raw data into the raw directory
    raw_directory = "output/baseline_generations_raw"
    if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
        json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

    # remove the thinking part of the generation
    for ques, inst in tqdm.tqdm(hint_data.items()):
        # final_hints = sent_tokenize('\n'.join(inst).split('</think>')[-1])
        # final_hints = '\n'.join(inst).split('</think>')[-1].split('\n')[1:]
        final_hints = inst
        # for idx, hint in enumerate(inst):
        #     if hint == '':
        #         final_hints = inst[idx+1:]
        #         break
        # print(final_hints)
        if len(final_hints) == 5 and (final_hints[0] in ["Hints:", '**Hints:**', '**Hints**:'] or final_hints[0].startswith('Here are four')):
            final_hints = final_hints[1:]
        if len(final_hints) == 3:
            final_hints.append(final_hints[2])
        final_hints = [clean_hint(hint, 4) for hint in final_hints]
        assert len(final_hints) == 4, f"Expected 4 hints, found {len(final_hints)} for instance {ques}\n{final_hints}"
        hint_data[ques] = final_hints
    
    # save the cleaned hints
    json.dump(hint_data, open(f'{output_dir}/{experiment_name}.json', 'w'), indent=4)

In [ ]:
# remove the gibberish preceeding the </think> tag from phi4

model_name = "phi4-14b"
print(f"Processing model: {model_name}")
data_split = 'valid'
experiment_name = f"{model_name}_static_{data_split}"
hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

# store the raw data into the raw directory
raw_directory = "output/baseline_generations_raw"
if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
    json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

# remove the thinking part of the generation
for ques, inst in tqdm.tqdm(hint_data.items()):
    final_hints = inst
    assert len(final_hints) == 4, f"Expected 4 hints, found {len(final_hints)} for instance {ques}\n{final_hints}"

## Dynamic Hints

In [ ]:
# remove the gibberish preceeding the hints for the gemma-3 model outputs

for model_name in ["gemma3-1b", "gemma3-4b", "gemma3-12b", "gemma3-27b"]:
    print(f"Processing model: {model_name}")
    data_split = 'valid'
    experiment_name = f'{model_name}_dynamic_{data_split}'
    hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

    # store the raw data into the raw directory
    raw_directory = "output/baseline_generations_raw"
    if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
        json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

    for ques, inst in hint_data.items():
        final_hints = [hint.split(':')[-1].replace('*', '').strip() for hint in inst]

        final_hints = [clean_hint(hint, 4) for hint in final_hints]

        assert len(final_hints) == 4, f"Expected 4 hints, found {len(inst)} for instance {ques}\n{final_hints}"
        hint_data[ques] = final_hints

    # save the cleaned hints
    json.dump(hint_data, open(f'{output_dir}/{experiment_name}.json', 'w'), indent=4)

In [ ]:
# remove the gibberish preceeding the </think> tag from qwen and deepseek model family outputs
for model_name in ["qwen3-0.6b", "qwen3-1.7b", "qwen3-4b", "qwen3-8b", "qwen3-14b", "qwen3-30b", "qwen3-32b"]:
    print(f"Processing model: {model_name}")
    data_split = 'valid'
    experiment_name = f"{model_name}_dynamic_{data_split}"
    hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

    # store the raw data into the raw directory
    raw_directory = "output/baseline_generations_raw"
    if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
        json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

    # remove the thinking part of the generation
    for ques, inst in hint_data.items():
        final_hints = inst
        while len(final_hints) < 4:
            final_hints.append(final_hints[-1])
        final_hints = [clean_hint(hint, 4) for hint in final_hints]
        assert len(final_hints) == 4, f"Expected 4 hints, found {len(final_hints)} for instance {ques}\n{final_hints}"
        hint_data[ques] = final_hints
    # save the cleaned hints
    json.dump(hint_data, open(f'{output_dir}/{experiment_name}.json', 'w'), indent=4)

In [ ]:
# remove the gibberish preceeding the </think> tag from qwen and deepseek model family outputs

for model_name in ["deepseek-r1-1.5b", "deepseek-r1-7b", "deepseek-r1-8b", "deepseek-r1-14b", "deepseek-r1-32b"]:
    print(f"Processing model: {model_name}")
    data_split = 'valid'
    experiment_name = f"{model_name}_dynamic_{data_split}"
    hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

    # store the raw data into the raw directory
    raw_directory = "output/baseline_generations_raw"
    if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
        json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

    # remove the thinking part of the generation
    for ques, inst in tqdm.tqdm(hint_data.items(), total=len(hint_data)):
        # final_hints = sent_tokenize('\n'.join(inst).split('</think>')[-1])
        final_hints = '\n'.join(inst).split('</think>')[-1].split('\n') # [1:]
        if final_hints[-1] == '```' :
            final_hints = final_hints[:-1]
        if final_hints[0].startswith('Here are four') or final_hints[0].startswith('The following are four') or final_hints[0].startswith('Certainly') or final_hints[0].startswith('Here are 4 hints') or final_hints[0].startswith('Here are the') or final_hints[0].startswith('Four hints that'):
            final_hints = final_hints[1:]
        if final_hints[0] in ["Hints:", '**Hints:**', '**Hints**:', '```plaintext', 'hints', 'hints:']:
            final_hints = final_hints[1:]
        if (final_hints[-1].startswith('Each hint') or final_hints[-1].startswith('These hints') or final_hints[-1].startswith("5.")):
            final_hints = final_hints[:-1]
        if len(final_hints) == 3:
            final_hints.append(final_hints[2])
        final_hints = [clean_hint(hint, 4) for hint in final_hints]
        if len(final_hints) > 4:
            final_hints = final_hints[:4]
        if len(final_hints) < 4:
            while len(final_hints) < 4:
                final_hints.append(final_hints[-1])
        assert len(final_hints) == 4, f"Expected 4 hints, found {len(final_hints)} for instance {ques}\n{final_hints}"
        hint_data[ques] = final_hints
    
    # save the cleaned hints
    json.dump(hint_data, open(f'{output_dir}/{experiment_name}.json', 'w'), indent=4)

In [ ]:
# remove the gibberish preceeding the </think> tag from phi4

model_name = "phi4-14b"
print(f"Processing model: {model_name}")
data_split = 'valid'
experiment_name = f"{model_name}_dynamic_{data_split}"
hint_data = json.load(open(f'{output_dir}/{experiment_name}.json', 'r'))

# store the raw data into the raw directory
raw_directory = "output/baseline_generations_raw"
if not os.path.exists(f'{raw_directory}/{experiment_name}.json'):
    json.dump(hint_data, open(f'{raw_directory}/{experiment_name}.json', 'w'), indent=4)

# remove the thinking part of the generation
for ques, inst in tqdm.tqdm(hint_data.items()):
    final_hints = inst
    assert len(final_hints) == 4, f"Expected 4 hints, found {len(final_hints)} for instance {ques}\n{final_hints}"